# 14 Environment and modules

<div class="bp-banner">
  <div class="bp-series">Introduction to the Bash Shell</div>
  <div style="display:flex;align-items:baseline;gap:14px;flex-wrap:wrap;">
    <span class="bp-title">Part IV — Working on a cluster</span>
    <span class="bp-meta">Notebook&nbsp;14</span>
  </div>
  <div style="margin-top:10px;max-width:62ch;color:#46506b;">
    The environment your shell runs inside (variables, <code>$PATH</code>,
    startup files, and loaded modules), and why a cluster job so often can't find
    the software you swear you set up.
  </div>
  <div class="bp-rule" style="display:flex;justify-content:space-between;flex-wrap:wrap;gap:8px;">
    <span class="bp-meta">Raymond Amador</span>
    <span class="bp-meta">v0.1.0&nbsp;·&nbsp;CC&nbsp;BY&nbsp;4.0 (text) / MIT (code)</span>
  </div>
</div>

In [1]:
# Hidden setup: stand at the repo root, source the validation gate, and turn on
# the (real) module system pointed at this repo's demo modulefiles. data/ is
# read-only; everything we make lives in a fresh scratch/.
ROOT="$PWD"; while [ ! -f "$ROOT/tools/check.sh" ] && [ "$ROOT" != "/" ]; do ROOT="$(dirname "$ROOT")"; done
source "$ROOT/tools/check.sh"
set +H
cd "$ROOT"
# environment-modules pages `avail`/`list` through a pager when it sees a terminal;
# the kernel runs in a pty, so without this it would hang on `less`. Force no pager.
export MODULES_PAGER=cat
for _init in /usr/share/modules/init/bash /etc/profile.d/modules.sh /opt/homebrew/opt/modules/init/bash; do
  [ -r "$_init" ] && { source "$_init"; break; }
done
export MODULEPATH="$ROOT/modulefiles"
module purge >/dev/null 2>&1 || true
true

## What this notebook is about

Part IV is about taking everything you have built (scripts that read, write,
decide, and repeat) to the place the real work happens: a shared cluster. And the
very first thing that goes wrong there is almost never your science. It is the
**environment**.

Your shell does not run in a vacuum. It runs inside a set of **variables** (and, on
a cluster, **loaded modules**) that every command and every script you launch
**inherits**. Getting that environment right is what makes the correct software
available; getting it subtly wrong is the single most common cluster bug, the one
behind the classic cry: *"but it worked when I ran it by hand!"*

This notebook is the map of that environment: shell variables versus exported ones
(§A), `$PATH` (§B), the startup files that set it all up, and the login-versus-batch
trap that breaks jobs (§C), the `source`-versus-run distinction that makes it all
work (§D), `alias` (§E), and finally the **module** system clusters use to hand you
software (§F). (As ever: **no physics**: we load a stand-in "code" without caring
what it computes.)

## A. Variables versus the environment

In Notebook 12 you set variables: `name=value`. Such a variable is **private to the
current shell**. The moment you run a script, you start a *child* shell, and the
child does **not** inherit your private variables. It inherits only the ones you
have **exported** into the **environment**. Here is that fact, felt. We have a tiny
child script that just reports one variable:

In [2]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch
printf '#!/usr/bin/env bash\necho "the child sees GREETING as: [${GREETING:-(unset)}]"\n' > scratch/child.sh
chmod +x scratch/child.sh

Set the variable the ordinary way and run the child: it sees **nothing**, because
a plain variable does not cross into the child:

In [3]:
GREETING="hello from the parent"; bash scratch/child.sh

the child sees GREETING as: [(unset)]


Now **`export`** it (promoting it from a shell variable to an *environment*
variable) and run the very same child again:

In [4]:
export GREETING; bash scratch/child.sh

the child sees GREETING as: [hello from the parent]


This time it crosses. That is the whole mental model of the environment: **a child
inherits exactly the exported variables, and nothing else.** Hold onto it: almost
every "my job can't see X" problem is this picture with X unexported.

```{command-card} export
```

The flip side of `export` is *seeing* what is in the environment, with **`printenv`**
(or the `env` command), and note that our newly-exported `GREETING` is now in it,
while a plain shell variable would not be:

```{command-card} printenv
```

In [5]:
printenv GREETING

hello from the parent


## B. `$PATH` revisited

You already met the most important environment variable in Notebook 11: **`$PATH`**,
the colon-separated list of directories the shell searches for a command you type by
name. It is just an environment variable, so you shape it with `export`. The
canonical move, the one that finally delivers Notebook 11's "promote your script to
a command," is to **prepend** your own `bin` directory:

```bash
export PATH="$HOME/bin:$PATH"
```

Prepending (your directory first) means your version wins when names collide; the
existing `$PATH` is kept on the end so every normal command still works. Watch it
make a script callable by bare name:

In [6]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch/bin
printf '#!/usr/bin/env bash\necho "greet-tool ran, found by name on PATH"\n' > scratch/bin/greet-tool
chmod +x scratch/bin/greet-tool

In [7]:
export PATH="$ROOT/scratch/bin:$PATH"; greet-tool

greet-tool ran, found by name on PATH


The bare name resolved (no `./`) because its directory is now on `$PATH`. One
rule guards this:

:::{admonition} ⚠ Extend PATH, never replace it
:class: warning
Always write `export PATH="newdir:$PATH"`, *keeping* the old `$PATH` on the end.
If you write a bare `export PATH="newdir"` you have **clobbered** it: the shell can
no longer find `ls`, `grep`, or anything else, and your session is effectively
broken until you fix it. The `:$PATH` on the end is not optional decoration; it is
the rest of your system.
:::

## C. Startup files: login versus non-login

Where do all these exports and PATH tweaks usually live? In a **startup file** that
the shell reads automatically. But *which* file, and *whether it is read at all*,
depends on the kind of shell, and this is the linchpin of the whole notebook.

<div class="bp-card">
  <span class="bp-card-cmd">Which startup file runs</span> — <span class="bp-card-job">depends on how the shell was started. This is the source of most "it works by hand but not in my job" bugs.</span>
  <table>
    <tr><td>~/.bash_profile&nbsp;·&nbsp;~/.profile</td><td><b>login</b> shells: when you <code>ssh</code> in or log in at a console</td></tr>
    <tr><td>~/.bashrc</td><td><b>non-login, interactive</b> shells: opening a new terminal tab</td></tr>
    <tr><td><i>(none)</i></td><td><b>non-interactive</b> shells (a script, or a batch job) read <b>no</b> rc file</td></tr>
  </table>
</div>

The common convention is that `~/.bash_profile` simply `source`s `~/.bashrc`, so a
login shell ends up with your interactive setup too. But look at that last row,
because it is the trap. The shell running *these very cells* is non-interactive,
exactly like a batch job; ask it:

In [8]:
case $- in
  *i*) echo "this shell is INTERACTIVE — it reads ~/.bashrc" ;;
  *)   echo "this shell is NON-interactive — like a script or a batch job, it reads NO rc file" ;;
esac

this shell is INTERACTIVE — it reads ~/.bashrc


:::{admonition} ⚠ Why your cluster job can't find your modules
:class: warning
A scheduler runs your job script in a **non-login, non-interactive** shell. So it
**never sources your `~/.bashrc`**: none of your aliases, your `$PATH` edits, or
your `module load` lines from there exist inside the job. This is *the* reason for
"it ran when I typed it, but the batch job says command not found." The fix is not
to fight the shell: it is to put the setup your job needs **inside the job script
itself** (Notebook 16), so it does not depend on being inherited.
:::

## D. `source` versus execute — the key distinction

That fix rests on one idea, and it is the most important tool in this notebook. When
you run `./script` (Notebook 11), it runs in a **child** shell, so anything it
changes (variables, `$PATH`, loaded modules) vanishes the instant it finishes. When
you **`source`** it instead, its lines run in your **current** shell, so its changes
**stay**. We have a script that sets a variable:

In [9]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch
printf '#!/usr/bin/env bash\nMSG="set inside the script"\n' > scratch/setvar.sh
chmod +x scratch/setvar.sh
unset MSG

Run it as a child with `./` (here, `bash …`) and the variable does **not** survive:
the child set it, then exited and took it with it:

In [10]:
bash scratch/setvar.sh; echo "after running it: [${MSG:-(unset)}]"

after running it: [(unset)]


Now **`source`** the same script: its line runs *here*, so the variable persists:

In [11]:
source scratch/setvar.sh; echo "after sourcing it: [$MSG]"

after sourcing it: [set inside the script]


```{command-card} source
```

That is exactly why you **`source ~/.bashrc`** to apply changes without opening a new
terminal, and why **`module load`** (below) has to alter your *current* shell to work
at all. Run vs. source is the difference between "did something and left" and
"changed where I'm standing."

## E. `alias` — interactive shorthand

A small convenience for the prompt: an **`alias`** gives a short name to a longer
command. The classic lives in everyone's `.bashrc`:

```{command-card} alias
```

In [12]:
alias ll='ls -lh'; alias ll

alias ll='ls -lh'


At an interactive prompt, typing `ll` now expands to `ls -lh`. But there is a catch
worth stating plainly, and it ties straight back to Notebook 12:

In [13]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch
printf '#!/usr/bin/env bash\nif command -v ll >/dev/null; then echo "the child script sees ll"; else echo "the child script does NOT see the ll alias"; fi\n' > scratch/tryalias.sh
chmod +x scratch/tryalias.sh

In [14]:
bash scratch/tryalias.sh

the child script does NOT see the ll alias


A script cannot see your aliases: they are an interactive-only convenience. So when
a *script* needs a reusable shorthand, you do not reach for an alias; you write a
**function** (Notebook 12), which a script defines for itself.

## F. The module system

On a cluster you do not install software, and it is not all sitting on `$PATH`
waiting for you. There are dozens of programs, often several **versions** of each
with conflicting dependencies, and you opt into exactly the ones you want with the
**module** system. It is the cluster embodiment of everything in this notebook:
`module load` works precisely by **modifying your current environment** (adding to
`$PATH`, setting variables) the way a `source`d script does.

```{command-card} module
```

Start by asking what is available. (We have set up a demo "simulation code" in two
versions, plus a second tool family; on a real cluster the list runs to hundreds.)
`module avail` writes its listing to standard error, which is normal:

In [15]:
module avail 2>&1

------------ /home/runner/work/bash-primer/bash-primer/modulefiles -------------


analysis-tools/3.2  democode/1.0  democode/2.0  


Key:


modulepath  


Before loading anything, the `democode` command does not exist: it is not on
`$PATH`:

In [16]:
command -v democode || echo "democode is not available yet"

democode is not available yet


Now **load** version 1.0. With no fanfare, it puts `democode` on your `$PATH` and
sets a variable, and suddenly the command runs:

In [17]:
module load democode/1.0; democode

democode 1.0 — stand-in simulation code (loaded via the module system)


In [18]:
echo "the module set DEMOCODE_VERSION=$DEMOCODE_VERSION"

the module set DEMOCODE_VERSION=1.0


**`module list`** shows what you have loaded, and **`module purge`** clears
everything, taking `democode` straight back off your `$PATH`:

In [19]:
module list 2>&1

Currently Loaded Modulefiles:


 1) democode/1.0  


In [20]:
module purge; command -v democode || echo "after purge, democode is gone again"

after purge, democode is gone again


That is the whole loop (`avail`, `load`, `list`, `purge`, plus `swap` to change
versions), and the reason it matters here: because a module only changes the
*current* environment, a batch job must run these `module load` lines **inside
itself**. That is exactly where Notebook 16 picks up.

## Exercises

Everything below is session-local and reversible: scratch files, a scratch `bin/`,
and module loads we purge afterwards. `data/` stays read-only. (Scripts are written
with here-documents for reproducibility; in your terminal you would use Vim.)

### Warm-up 1 (worked) — Inheritance, felt

Set a variable, run a child script that reads it: first **unexported** (the child
sees nothing), then **exported** (the child sees it).

In [21]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch
printf '#!/usr/bin/env bash\necho "[${LABEL:-(unset)}]"\n' > scratch/show.sh
chmod +x scratch/show.sh
unset LABEL

In [22]:
# (solution hidden on the public site)


unexported:


[(unset)]


exported:


[lj38]


In [23]:
unset LABEL; LABEL="lj38"
unexp="$(bash "$ROOT/scratch/show.sh")"
export LABEL
exp="$(bash "$ROOT/scratch/show.sh")"
unset LABEL
check '[ "$unexp" = "[(unset)]" ] && [ "$exp" = "[lj38]" ]' \
      "the child saw nothing until the variable was exported"

✓ the child saw nothing until the variable was exported


### Warm-up 2 (your turn) — Extend `$PATH`

Put a script in a scratch `bin/`, prepend that directory to `$PATH`, and call the
script by its **bare name** (no `./`).

In [24]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch/bin
printf '#!/usr/bin/env bash\necho "mytool reporting in"\n' > scratch/bin/mytool
chmod +x scratch/bin/mytool

In [25]:
# (solution hidden on the public site)


mytool reporting in


In [26]:
out="$(PATH="$ROOT/scratch/bin:$PATH" mytool)"
check 'printf "%s" "$out" | grep -q "mytool reporting in"' \
      "the script ran by bare name once its directory was on PATH"

✓ the script ran by bare name once its directory was on PATH


### Applied 1 (your turn) — `source` versus `./`

A script sets a variable. Run it with `./` (a child, where the variable does not persist)
and then `source` it (it does). Report the variable after each.

In [27]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch
printf '#!/usr/bin/env bash\nTOOL_HOME="/opt/democode"\n' > scratch/conf.sh
chmod +x scratch/conf.sh
unset TOOL_HOME

In [28]:
# (solution hidden on the public site)


after ./:     [(unset)]


after source: [/opt/democode]


In [29]:
unset TOOL_HOME
bash "$ROOT/scratch/conf.sh"
child="${TOOL_HOME:-(unset)}"
source "$ROOT/scratch/conf.sh"
sourced="${TOOL_HOME:-(unset)}"
unset TOOL_HOME
check '[ "$child" = "(unset)" ] && [ "$sourced" = "/opt/democode" ]' \
      "the variable persisted only after source, not after running as a child"

✓ the variable persisted only after source, not after running as a child


### Applied 2 (your turn) — Load software with `module`

List what is available, load `democode/1.0`, confirm it is on `$PATH` and listed,
then `purge` and confirm it is gone.

In [30]:
cd "$ROOT"; module purge >/dev/null 2>&1 || true

In [31]:
# (solution hidden on the public site)


------------ /home/runner/work/bash-primer/bash-primer/modulefiles -------------


analysis-tools/3.2  democode/1.0  democode/2.0  


Key:


modulepath  


loaded; democode is at: /home/runner/work/bash-primer/bash-primer/opt/democode-1.0/bin/democode


Currently Loaded Modulefiles:


 1) democode/1.0  


after purge: none


In [32]:
module purge >/dev/null 2>&1
before="$(command -v democode || echo none)"
module load democode/1.0
after="$(command -v democode || echo none)"
ver="${DEMOCODE_VERSION:-none}"
module purge >/dev/null 2>&1
gone="$(command -v democode || echo none)"
check '[ "$before" = none ] && [ "$after" != none ] && [ "$ver" = "1.0" ] && [ "$gone" = none ]' \
      "load put democode on PATH and set its version; purge removed it"

✓ load put democode on PATH and set its version; purge removed it


### Composite — putting it together (prepare an environment)

The capstone, and the shape of every cluster setup you will ever write. Compose one
small **env-setup file** that does three things (`export`s a variable, extends
`$PATH` with your scratch `bin/`, and `module load`s the demo code), then **`source`**
it and confirm that *both* your own tool and the loaded code now run. (In Notebook
16 this exact block moves inside a job script.)

In [33]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch/bin; module purge >/dev/null 2>&1 || true
printf '#!/usr/bin/env bash\necho "analyze: processing $PROJECT with democode"\n' > scratch/bin/analyze
chmod +x scratch/bin/analyze
unset PROJECT

In [34]:
# (solution hidden on the public site)


analyze: processing lj38 with democode


democode 1.0 — stand-in simulation code (loaded via the module system)


In [35]:
module purge >/dev/null 2>&1; unset PROJECT
result="$( source "$ROOT/scratch/env.sh" >/dev/null 2>&1; analyze; democode; echo "PROJECT=$PROJECT" )"
module purge >/dev/null 2>&1
check 'printf "%s" "$result" | grep -q "analyze: processing lj38" && printf "%s" "$result" | grep -q "democode 1.0" && printf "%s" "$result" | grep -q "PROJECT=lj38"' \
      "sourcing the env file exported the variable, extended PATH, and loaded the code — all three tools then ran"

✓ sourcing the env file exported the variable, extended PATH, and loaded the code — all three tools then ran


### Optional stretch (your turn) — A `.bashrc`-style file

Build a scratch rc file with an **export**, a **function**, and an **alias**;
`source` it; and confirm the export and the function work, while a child **script**
cannot see the alias (so a script would use the function instead).

In [36]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch; unset EDITOR
printf '#!/usr/bin/env bash\ncommand -v ll >/dev/null && echo "child: sees ll" || echo "child: no ll alias"\n' > scratch/probe.sh
chmod +x scratch/probe.sh

In [37]:
# (solution hidden on the public site)


EDITOR is now: vim


hello from a function


alias ll='ls -lh'


child: no ll alias


In [38]:
unset EDITOR; unset -f greet 2>/dev/null
source "$ROOT/scratch/myrc.sh"
g="$(greet)"
probe="$(bash "$ROOT/scratch/probe.sh")"
check '[ "$EDITOR" = "vim" ] && [ "$g" = "hello from a function" ] && printf "%s" "$probe" | grep -q "no ll alias"' \
      "source applied the export and the function; the child script could not see the alias"

✓ source applied the export and the function; the child script could not see the alias


## Outlook

You can now read and shape the environment your scripts run in (exported variables,
`$PATH`, startup files, `source` versus run, and the module system), and, crucially,
you know *why* a batch job may inherit none of it. That last point is the bridge to
the rest of Part IV. Next (Notebook 15): actually getting onto a remote machine and
moving your scripts and data there: `ssh`, `scp`, `rsync`, archiving with `tar`, and
keeping a long job alive after you log out.

```{compendium-new}
```

<div class="bp-banner" style="margin-top:30px;">
  <div class="bp-series">Take this notebook with you</div>
  <div style="font-size:14.5px;line-height:1.55;max-width:66ch;">
    Open a <b>live terminal</b> from the &ldquo;Practice here&rdquo; box in any
    section to export, source, and <code>module load</code> for yourself; the
    demo modules are real and installed. The published notebooks ship
    <b>without worked solutions</b>; if you would like the reference solutions
    (to teach from or to check your own work), get in touch:
    <a href="mailto:hello@ramador.me">hello@ramador.me</a>.
  </div>
</div>